In [14]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from collections import Counter

In [9]:
dirPath = "..\\..\\Dataset\\train\\annos"

In [10]:
print(os.listdir(dirPath)[63])

with open(os.path.join(dirPath, os.listdir(dirPath)[63]), 'r', encoding='utf-8') as file:
    d = json.load(file)

print(d)

000064.json
{'item2': {'segmentation': [[170, 361, 241, 369, 316, 365, 333, 460, 351, 541, 225, 557, 146, 516, 147, 430, 170, 361]], 'scale': 2, 'viewpoint': 2, 'zoom_in': 1, 'landmarks': [170, 361, 1, 241, 369, 1, 316, 365, 1, 147, 430, 2, 146, 516, 2, 225, 557, 2, 351, 541, 2, 333, 460, 2], 'style': 0, 'bounding_box': [131, 361, 357, 577], 'category_id': 9, 'occlusion': 2, 'category_name': 'skirt'}, 'source': 'user', 'pair_id': 7, 'item1': {'segmentation': [[284, 178, 276, 203, 242, 211, 216, 199, 208, 175, 147, 210, 137, 257, 129, 305, 170, 314, 171, 297, 171, 281, 173, 302, 170, 338, 168, 373, 247, 400, 317, 384, 319, 345, 316, 305, 317, 291, 317, 308, 319, 320, 369, 317, 363, 266, 344, 209, 284, 178], [147, 210, 137, 257, 129, 305, 170, 314, 171, 297, 171, 281, 147, 210], [317, 291, 317, 308, 319, 320, 369, 317, 363, 266, 344, 209, 317, 291]], 'scale': 2, 'viewpoint': 2, 'zoom_in': 1, 'landmarks': [241, 186, 1, 208, 175, 2, 216, 199, 2, 242, 211, 2, 276, 203, 2, 284, 178, 1, 147, 

In [11]:
#To understand how each json file is structured
print(os.listdir(dirPath)[63])

with open(os.path.join(dirPath, os.listdir(dirPath)[63]), 'r', encoding='utf-8') as file:
    data = json.load(file)

print(data.keys())
print()
print(data['item1'].keys())

keys = list(data.keys())
sorted(keys)
print(keys)

c = {}
for i in range(len(keys)-3):
    pair = tuple([ data[keys[i]]['category_id'], data[keys[i]]['category_name'] ])

    if pair not in c: c[pair] = 1
    else: c[pair] += 1

print(c)

#Strategy: Load json, sort the keys, iterate through all item* dicts, extract (category_id, category_name) and update count

000064.json
dict_keys(['item2', 'source', 'pair_id', 'item1'])

dict_keys(['segmentation', 'scale', 'viewpoint', 'zoom_in', 'landmarks', 'style', 'bounding_box', 'category_id', 'occlusion', 'category_name'])
['item2', 'source', 'pair_id', 'item1']
{(9, 'skirt'): 1}


In [ ]:
classes = Counter()

for path in os.listdir(dirPath): #1,91,961 files present in train folder
    filePath = os.path.join(dirPath, path)
    with open(filePath, 'r', encoding='utf-8') as file:
        data = json.load(file)

    keys = list(data.keys())
    keys.sort()
    
    for i in range(len(keys)-3):
        pair = tuple([ data[keys[i]]['category_id'], data[keys[i]]['category_name'] ])

        if pair not in classes: classes[pair] = 1
        else: classes[pair] += 1

toConsider = classes.most_common(5)
for (cat_id, cat_name), count in toConsider:
    print(f"ID: {cat_id} Name: {cat_name} Count: {count}")

ID: 1 Name: short sleeve top Count: 43511
ID: 2 Name: long sleeve top Count: 18285
ID: 8 Name: trousers Count: 17447
ID: 5 Name: vest Count: 10562
ID: 7 Name: shorts Count: 9466


In [15]:
#Looking at the structure of JSON, all info will be needed for training. 
#Strategy: Load json, sort the keys, iterate through all item* dicts, extract all info and save to a dataframe, then to CSV

infoList = []

for path in os.listdir(dirPath): #1,91,961 files present in train folder
    filePath = os.path.join(dirPath, path)
    with open(filePath, 'r', encoding='utf-8') as file:
        data = json.load(file) 

    for key in data.keys():
        if key.startswith('item'):
            row = {
                'file_name': path,
                'item_id': key,
                'category_id': data[key]['category_id'],
                'category_name': data[key]['category_name'],
                'bbox': data[key]['bounding_box'], 
                'segmentation': data[key]['segmentation'],
                'landmarks': data[key]['landmarks'],
                'style': data[key]['style'],
                'occlusion': data[key]['occlusion']
            }

            if(data[key]['category_id'] in toConsider): row['isConsidered'] = True
            else: row['isConsidered'] = False

            infoList.append(row)

infoDf = pd.DataFrame(infoList)
infoDf.to_csv('..\\..\\Dataset\\processed\\per_item_info.csv', index=False)

In [16]:
item_cols = ['item_id', 'category_id', 'category_name', 'bbox', 'segmentation', 'landmarks', 'style', 'occlusion', 'isConsidered']

imageDF = infoDf.groupby('file_name').apply(lambda x: x[item_cols].to_dict('records')).reset_index()
imageDF.columns = ['file_name', 'items_info']

imageDF.to_csv('..\\..\\Dataset\\processed\\per_image_info.csv', index=False)